# Parsing Files into Blocks with Padding

[Secure Hash Standard, Section 5.1.1](https://doi.org/10.6028/NIST.FIPS.180-4)

[Real Python: *How to Use Generators and yield in Python*](https://realpython.com/introduction-to-python-generators/)

## File to Blocks 

In [1]:
def file_to_blocks(filepath, no_bytes=64):
  """Read a file in no_bytes chunks."""
  # Open the file in binary mode.
  f = open(filepath, 'rb')
  # Yield each new block of the file.
  while block := f.read(no_bytes):
      yield block

In [2]:
# Example instance.
G = file_to_blocks('padding.ipynb')

# G is a generator.
G

<generator object file_to_blocks at 0x107a62500>

In [3]:
# Get the first block.
block = next(G)

# Show - note the b at start of output.
block

b'{\n "cells": [\n  {\n   "cell_type": "markdown",\n   "id": "af2ef237'

In [4]:
# block is a bytes object.
type(block)

bytes

In [5]:
# The fourth byte of block in hex.
hex(block[3])

'0x22'

In [6]:
# Print all the bytes in hex.
' '.join([f'{b:02X}' for b in block])

'7B 0A 20 22 63 65 6C 6C 73 22 3A 20 5B 0A 20 20 7B 0A 20 20 20 22 63 65 6C 6C 5F 74 79 70 65 22 3A 20 22 6D 61 72 6B 64 6F 77 6E 22 2C 0A 20 20 20 22 69 64 22 3A 20 22 61 66 32 65 66 32 33 37'

In [7]:
# Make it a function.
def bytes_to_hex(B, sep=' '):
  return sep.join([f'{b:02X}' for b in B])

# Example.
bytes_to_hex(block)

'7B 0A 20 22 63 65 6C 6C 73 22 3A 20 5B 0A 20 20 7B 0A 20 20 20 22 63 65 6C 6C 5F 74 79 70 65 22 3A 20 22 6D 61 72 6B 64 6F 77 6E 22 2C 0A 20 20 20 22 69 64 22 3A 20 22 61 66 32 65 66 32 33 37'

In [8]:
# Count the blocks (without padding) in the file.

# A counter.
no_blocks = 0

# Keep reading blocks until they're all used.
for block in file_to_blocks('padding.ipynb'):
  # Increment counter for each block.
  no_blocks = no_blocks + 1

# Show number of blocks.
no_blocks

288

In [9]:
# Count the number of bytes.

# Counter.
no_bytes = 0

# Loop through the file in blocks.
for block in file_to_blocks('padding.ipynb'):
  # Count the number of bytes returned.
  no_bytes = no_bytes + len(block)

# Show.
no_bytes

18384

A smaller file - just 3 bytes.

In [10]:
# Count the number of bytes.

# Counter.
no_bytes = 0

# Loop through the file in blocks.
for block in file_to_blocks('data/abc.txt'):
  # Count the number of bytes returned.
  no_bytes = no_bytes + len(block)

# Show.
no_bytes

3

In [11]:
# Count the blocks (without padding) in the file.

# A counter.
no_blocks = 0

# Keep reading blocks until they're all used.
for block in file_to_blocks('data/abc.txt'):
  # Increment counter for each block.
  no_blocks = no_blocks + 1

# Show number of blocks.
no_blocks

1

An empty file - 0 bytes.

In [12]:
# Count the number of bytes.

# Counter.
no_bytes = 0

# Loop through the file in blocks.
for block in file_to_blocks('data/empty.txt'):
  # Count the number of bytes returned.
  no_bytes = no_bytes + len(block)

# Show.
no_bytes

0

In [13]:
# Count the blocks (without padding) in the file.

# A counter.
no_blocks = 0

# Keep reading blocks until they're all used.
for block in file_to_blocks('data/empty.txt'):
  # Increment counter for each block.
  no_blocks = no_blocks + 1

# Show number of blocks.
no_blocks

0

## Big and Little Endian

https://docs.python.org/3/library/stdtypes.html#int.to_bytes

In [14]:
# A large integer.
val =  0x123456789ABCDEF0

# In big endian.
print("Big endian:")
print(' '.join([f"{i:02X}" for i in val.to_bytes(8, byteorder='big')]))

# In little endian.
print("Little endian:")
print(' '.join([f"{i:02X}" for i in val.to_bytes(8, byteorder='little')]))

Big endian:
12 34 56 78 9A BC DE F0
Little endian:
F0 DE BC 9A 78 56 34 12


## `bytes` Objects

[Using `+` for concatenation](https://realpython.com/python-bytes/#frequently-asked-questions:~:text=Just%20like%20strings%2C%20bytes%20objects%20support%20two%20additional%20operators.%20The%20star%20operator%20(*)%20allows%20you%20to%20repeat%20the%20same%20byte%20sequence%20multiple%20times%2C%20while%20the%20plus%20operator%20(%2B)%20lets%20you%20concatenate%20two%20or%20more%20bytes%20instances%20into%20one%3A)

In [15]:
# Bytes objects can be joined using +.
bytes([0xff, 0xf0]) +  bytes([0x0f, 0x00])

b'\xff\xf0\x0f\x00'

In [16]:
# Gives a bytes object with 9 zero bytes.
bytes([0x00] * 9)

b'\x00\x00\x00\x00\x00\x00\x00\x00\x00'

## Padding

In [17]:
def padded_message_generator(filepath):
    """Return blocks of the file at filepath, padded."""
    # Number of bits read.
    no_bits = 0
    
    # Keep reading blocks until they're all used.
    for block in file_to_blocks(filepath):
      # Count the number of bytes returned.
      no_bits = no_bits + (len(block) * 8)
      # If we read a full block, yield it.
      if len(block) == 64:
         yield block

    # We use block below, so we need to make sure it has a value.
    # If the file is empty, it won't.
    if no_bits == 0:
      block = bytes()

    # Once we get here, there are three scenarios.
    # 1. There are at least 9 bytes available in the current block.
    # 2. There are not at least 9 bytes available but there is at least 1 byte.
    # 3. There were no bytes available in the last block.

    # Scenario 1: are there at least 9 bytes available?
    if (64 - len(block)) >= 9:
       yield (block
              + bytes([0x80])
              + bytes([0x00] * (64 - len(block) - 1 - 8))
              + no_bits.to_bytes(8, byteorder='big'))
    # Scenario 2: there are between 8 and 1 bytes available, inclusive.
    elif (64 - len(block)) >= 1:
       yield block + bytes([0x80]) + bytes([0x00] * (64 - len(block) - 1))
       yield bytes([0x00] * 56) + no_bits.to_bytes(8, byteorder='big')
    # Scenario 3: There were no bytes available in the last block.
    else:
       yield bytes([0x80] + ([0x00] * 55)) + no_bits.to_bytes(8, byteorder='big')

Example large file.

In [18]:
# Counter for blocks.
no_blocks = 0

# Loop through blocks, counting them.
for block in padded_message_generator('padding.ipynb'):
    no_blocks = no_blocks + 1

# Show.
no_blocks, bytes_to_hex(block), len(block)

(288,
 '6D 61 74 5F 6D 69 6E 6F 72 22 3A 20 35 0A 7D 0A 80 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 02 3E 80',
 64)

Example small file.

In [19]:
# Counter for blocks.
no_blocks = 0

# Loop through blocks, counting them.
for block in padded_message_generator('data/one-two-eight.txt'):
    no_blocks = no_blocks + 1

# Show.
no_blocks, bytes_to_hex(block)

(3,
 '80 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 04 00')

Example smaller file.

In [20]:
# Counter for blocks.
no_blocks = 0

# Loop through blocks, counting them.
for block in padded_message_generator('data/abc.txt'):
    no_blocks = no_blocks + 1

# Show.
no_blocks, bytes_to_hex(block)

(1,
 '61 62 63 80 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 18')

Example empty file.

In [21]:
# Counter for blocks.
no_blocks = 0

# Loop through blocks, counting them.
for block in padded_message_generator('data/empty.txt'):
    no_blocks = no_blocks + 1

# Show.
no_blocks, bytes_to_hex(block)

(1,
 '80 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00')

## End